In [0]:
yourfile='data_f0051149-9530-4309-8145-ee2f3bcaf5e4_86665c09-99b2-4e53-852b-1f94b78b1b7f.txt'
df = spark.read.option("header", "true").format("csv").load(f"abfss://raw@bhavanmetadata.dfs.core.windows.net/{yourfile}")
df.show(truncate=True)
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("ram.default.todayjob")

### 
**Find the employees whoes salary got reduced in there tenure atleast once**

In [0]:
data = [
    (101, "Ravi",  "2021-01-10", "2021-12-31", 30000),
    (101, "Ravi",  "2022-01-01", "2022-12-31", 35000),
    (101, "Ravi",  "2023-01-01", "2023-12-31", 33000),
    (101, "Ravi",  "2024-01-01", "2024-12-31", 37000),

    (102, "Priya", "2020-06-15", "2021-06-30", 40000),
    (102, "Priya", "2021-07-01", "2022-06-30", 45000),
    (102, "Priya", "2022-07-01", "2023-06-30", 42000),

    (103, "Arjun", "2019-03-01", "2020-02-29", 28000),
    (103, "Arjun", "2020-03-01", "2021-02-28", 30000),
    (103, "Arjun", "2021-03-01", "2022-02-28", 32000),

    (104, "Sneha", "2022-05-20", "2023-05-19", 50000),
    (104, "Sneha", "2023-05-20", "2024-05-19", 48000),

    (105, "Kiran", "2021-08-01", "2022-07-31", 55000),
    (105, "Kiran", "2022-08-01", "2023-07-31", 60000),
    (105, "Kiran", "2023-08-01", "2024-07-31", 58000),

    (106, "Meena", "2020-11-10", "2021-11-09", 37000),
    (106, "Meena", "2021-11-10", "2022-11-09", 39000),
    (106, "Meena", "2022-11-10", "2023-11-09", 41000)
]

columns = ["emp_id", "emp_name", "start_date", "end_date", "salary"]

df = spark.createDataFrame(data, columns)

df.show(truncate=False)

In [0]:
from pyspark.sql.functions import col

df = df.withColumn("start_date", col("start_date").cast("date")) \
       .withColumn("end_date", col("end_date").cast("date"))

In [0]:
df.printSchema()

In [0]:
df.createOrReplaceTempView("df")


In [0]:
%sql
with k as (
    select *,dense_rank() over(partition by emp_id order by salary) sal_ord,
    lead(start_date) over(partition by emp_id order by start_date) srt_ord
    ,dense_rank() over(partition by emp_id order by start_date) sal_dat
from df
)
select distinct emp_id, emp_name from k
where sal_ord!=sal_dat



Just tried this way (This is not the solution)

In [0]:
%sql
select emp_id,emp_name,start_date,salary,lead(start_date) over (partition by emp_id order by start_date) as next_start_date,
lead(salary) over (partition by emp_id order by start_date) next_salary from (with k as (select emp_id id,max(salary) mx_salary,min(salary) mn_salary from df group by emp_id),
mx as (select * from df
join k on(k.id=df.emp_id and k.mx_salary=df.salary)),
mn as (select * from df 
join k on(k.id=df.emp_id and k.mn_salary=df.salary))
select * from mx
union all
select * from mn)




**Flatten the list**

In [0]:
def ram():
    l=[1,2,[3,4],6,7]
    k=[]
    for i in l:
        if isinstance(i, list):
            k.extend(i)
        else:
            k.append(i)

    return (str(k).replace(" ", ""))
print(ram())